# Stage 4 — Phase 5.2/5.3: Fine-Tuning Timing Probe + Training

Uses `src/stage4_symbol_detection/train.py`. Starts with **Qwen3-VL** to prove the
pipeline end-to-end (measured timing probe, not the hand-estimate in
Stage4_Checklist_Status.md), then repeats for InternVL3 and Molmo2-O-7B.

Per the 2026-07-10 resequencing decision: this trains a SINGLE combined LoRA
directly on the detection task (5.2+5.3 merged), not a separate domain-adapt-then-
adapter sequence. The proper task-neutral domain-adapt pass is deferred to
end-of-Stage-4, as its own separate checkpoint.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os
if os.path.exists('/content/pid-ml'):
    !cd /content/pid-ml && git pull
else:
    !git clone https://github.com/TomGeorge45/pid-opensrc-substitution.git /content/pid-ml

## 3. Install dependencies

`peft` is new — needed for LoRA. Using the transformers version already verified working
for Qwen3-VL/InternVL3 (5.x line) for this first probe.

In [ ]:
!pip install -q transformers accelerate peft pycocotools supervision \
    kagglehub kaggle qwen-vl-utils einops timm

## 4. Path setup + imports

In [ ]:
import sys
sys.path.insert(0, '/content/pid-ml/src')

from stage4_symbol_detection import qwen_candidate, train

## 5. Load Qwen3-VL and build training examples

In [ ]:
processor, model = qwen_candidate.load()

examples = train.build_examples("qwen3vl")
print(f"Total training examples: {len(examples)}")
n_gupta = sum(1 for e in examples if e['source'] == 'gupta')
n_kaggle = sum(1 for e in examples if e['source'] == 'kaggle')
print(f"  gupta tiles: {n_gupta} | kaggle images: {n_kaggle}")

## 6. Apply LoRA

In [ ]:
model = train.setup_lora(model)

## 7. Timing probe — MEASURED, not estimated

This is the number that actually matters for planning — everything in
Stage4_Checklist_Status.md before this point is a hand-wavy guess.

In [ ]:
avg_step_time = train.time_training_steps(
    processor, model, qwen_candidate.SYMBOL_DETECTION_PROMPT, examples,
    n_steps=5, image_first=True,
)

n_examples = len(examples)
for n_epochs in (1, 2, 3):
    total_s = avg_step_time * n_examples * n_epochs
    print(f"{n_epochs} epoch(s): {total_s/3600:.2f}h projected (measured {avg_step_time:.2f}s/step x {n_examples} examples)")